# 📖 How2Sign Continuous Training Pipeline

**Purpose:** Train a Continuous Sign Language Recognition (CSLR) model using the How2Sign dataset.
This pipeline loads pre-extracted OpenPose keypoints from `.json` files and trains a Seq2Seq BiLSTM to output sequences of words.

In [2]:
from pathlib import Path
# ============================================================
# 2. DATASET PATHS  (local – How2Sign-keypoints dataset)
# ============================================================
BASE_DIR = Path(r'/kaggle/input/datasets/hadeelgamal/artifacts2')

CSV_TRAIN = Path(r'/kaggle/input/datasets/hadeelgamal/artifacts2/how2sign_train_subset.csv')
CSV_VAL   = Path(r'/kaggle/input/datasets/hadeelgamal/artifacts2/how2sign_val_subset.csv')
CSV_TEST  = Path(r'/kaggle/input/datasets/hadeelgamal/artifacts2/how2sign_test_subset.csv')


JSON_DIR_TRAIN = Path(r'/kaggle/input/datasets/hadeelgamal/artifacts2/train_features')
JSON_DIR_VAL   = Path(r'/kaggle/input/datasets/hadeelgamal/artifacts2/val_features')
JSON_DIR_TEST  = Path(r'/kaggle/input/datasets/hadeelgamal/artifacts2/test_features')

for label, p in [('BASE_DIR', BASE_DIR),
                 ('CSV_TRAIN', CSV_TRAIN), ('CSV_VAL', CSV_VAL), ('CSV_TEST', CSV_TEST),
                 ('JSON_TRAIN', JSON_DIR_TRAIN), ('JSON_VAL', JSON_DIR_VAL), ('JSON_TEST', JSON_DIR_TEST)]:
    print(f"{label:12s} exists={p.exists()}  → {p}")



BASE_DIR     exists=True  → /kaggle/input/datasets/hadeelgamal/artifacts2
CSV_TRAIN    exists=True  → /kaggle/input/datasets/hadeelgamal/artifacts2/how2sign_train_subset.csv
CSV_VAL      exists=True  → /kaggle/input/datasets/hadeelgamal/artifacts2/how2sign_val_subset.csv
CSV_TEST     exists=True  → /kaggle/input/datasets/hadeelgamal/artifacts2/how2sign_test_subset.csv
JSON_TRAIN   exists=True  → /kaggle/input/datasets/hadeelgamal/artifacts2/train_features
JSON_VAL     exists=True  → /kaggle/input/datasets/hadeelgamal/artifacts2/val_features
JSON_TEST    exists=True  → /kaggle/input/datasets/hadeelgamal/artifacts2/test_features


## 2.6  Sequence Length Analysis
Scan the extracted `.npy` files to find the real frame-length distribution.
`SEQUENCE_LENGTH` is set to the **95th-percentile** so that >95 % of sentences
are fully preserved, with no arbitrary hard cap.


In [2]:
# ============================================================
# 2.6  SEQUENCE LENGTH ANALYSIS  (auto-sets SEQUENCE_LENGTH)
# ============================================================
import os, glob
import numpy as np
def analyse_lengths(features_dirs):
    """
    Scan all .npy files across the supplied directories.
    Returns a sorted array of frame counts.
    """
    lengths = []
    for d in features_dirs:
        for fp in glob.glob(os.path.join(d, '*.npy')):
            try:
                arr = np.load(fp, mmap_mode='r')   # no full load needed
                lengths.append(arr.shape[0])
            except Exception:
                pass
    return np.array(lengths, dtype=np.int32)
FEATURE_DIRS = [BASE_DIR / 'train_features', BASE_DIR / 'val_features', BASE_DIR / 'test_features']
lengths = analyse_lengths(FEATURE_DIRS)
if len(lengths) == 0:
    # Fallback: nothing extracted yet — use a conservative default
    print("⚠️  No .npy files found yet. Defaulting SEQUENCE_LENGTH = 300.")
    print("   Re-run this cell after Section 2.5 has extracted the features.")
    SEQUENCE_LENGTH = 300
else:
    p50  = int(np.percentile(lengths, 50))
    p90  = int(np.percentile(lengths, 90))
    p95  = int(np.percentile(lengths, 95))
    p99  = int(np.percentile(lengths, 99))
    pmax = int(lengths.max())
    pmin = int(lengths.min())
    pmean= int(lengths.mean())
    print(f"Frame-length distribution across {len(lengths):,} clips:")
    print(f"  Min    : {pmin:>5} frames")
    print(f"  Mean   : {pmean:>5} frames")
    print(f"  P50    : {p50:>5} frames   (median)")
    print(f"  P90    : {p90:>5} frames")
    print(f"  P95    : {p95:>5} frames   ← chosen as SEQUENCE_LENGTH")
    print(f"  P99    : {p99:>5} frames")
    print(f"  Max    : {pmax:>5} frames")
    # Round up to nearest multiple of 8 (cleaner for GPU batching)
    SEQUENCE_LENGTH = int(np.ceil(p95 / 8) * 8)
    pct_preserved = (lengths <= SEQUENCE_LENGTH).mean() * 100
    print(f"\nSEQUENCE_LENGTH = {SEQUENCE_LENGTH}  "
          f"(P95 rounded to ×8 — preserves {pct_preserved:.1f}% of clips fully)")
    if SEQUENCE_LENGTH < 200:
        print("\n⚠️  WARNING: SEQUENCE_LENGTH < 200. Many sentences may still "
              "be truncated.\n   Consider using P99 instead.")


NameError: name 'BASE_DIR' is not defined

## 3. Custom Data Generator
This generator reads the OpenPose JSON files batch-by-batch to prevent out-of-memory errors.

In [4]:
# ============================================================
# 3.  DATA GENERATOR  (reads .npy — 232-dim)
# ============================================================
from tensorflow.keras.utils import Sequence
import pandas as pd
# SEQUENCE_LENGTH is set dynamically by Section 2.6 above
# (95th-percentile of actual extracted clip lengths)
SEQUENCE_LENGTH = SEQUENCE_LENGTH  # noqa: F841 — defined in cell 2.6
NUM_FEATURES    = 232   # body(50) + lhand(42) + rhand(42) + face(98)
class How2SignGenerator(Sequence):
    COL_NAME     = 'SENTENCE_NAME'
    COL_NPY      = 'NPY_PATH'
    COL_SENTENCE = 'SENTENCE'
    def __init__(self, csv_path, json_dir=None,
                 tokenizer=None,
                 batch_size      : int = 8,
                 sequence_length : int = SEQUENCE_LENGTH,
                 num_features    : int = NUM_FEATURES):
        csv_path = Path(csv_path) if csv_path else None
        if csv_path is None or not csv_path.exists():
            print(f'⚠️  CSV not found: {csv_path}')
            self.df = pd.DataFrame()
        else:
            self.df = pd.read_csv(csv_path, on_bad_lines='skip')
            missing = {self.COL_NAME, self.COL_NPY} - set(self.df.columns)
            if missing:
                print(f'⚠️  Missing columns: {missing}')
                self.df = pd.DataFrame()
            else:
                self.df = (self.df[self.df[self.COL_NPY].notna()]
                               .reset_index(drop=True))
                print(f'✅ {len(self.df):,} rows  ←  {csv_path.name}')
                print(f'   Feature dim : {num_features}')
                print(f'   Seq length  : {sequence_length}')
        self.tokenizer       = tokenizer
        self.batch_size      = batch_size
        self.sequence_length = sequence_length
        self.num_features    = num_features
    def _load_npy(self, npy_path: str) -> np.ndarray:
        """
        Load (T, 232) array → pad/truncate to
        (sequence_length, num_features).
        """
        out = np.zeros((self.sequence_length, self.num_features),
                       dtype=np.float32)
        try:
            full_path = str(BASE_DIR / npy_path)
            seq = np.load(full_path)                     # (T, 232)
            T   = min(len(seq), self.sequence_length)
            out[:T] = seq[:T, :self.num_features]
        except Exception as e:
            print(f'⚠️  {full_path}: {e}')
        return out
    def __len__(self):
        return max(1, int(np.ceil(len(self.df) / self.batch_size)))
    def __getitem__(self, idx):
        batch = self.df.iloc[idx * self.batch_size :
                             (idx + 1) * self.batch_size]
        X = np.zeros((len(batch), self.sequence_length, self.num_features),
                     dtype=np.float32)
        for i, (_, row) in enumerate(batch.iterrows()):
            X[i] = self._load_npy(str(row[self.COL_NPY]))
        sentences = (batch[self.COL_SENTENCE].fillna('')
                     if self.COL_SENTENCE in batch.columns
                     else pd.Series(['']*len(batch))).tolist()
        return X, sentences
    def on_epoch_end(self):
        self.df = self.df.sample(frac=1).reset_index(drop=True)
# ── Instantiate ──────────────────────────────────────────────
train_gen = How2SignGenerator(CSV_TRAIN, batch_size=8)
val_gen   = How2SignGenerator(CSV_VAL,   batch_size=8)
test_gen  = How2SignGenerator(CSV_TEST,  batch_size=8)
print(f'\nTrain batches : {len(train_gen)}')
print(f'Val   batches : {len(val_gen)}')
print(f'Test  batches : {len(test_gen)}')
if len(train_gen) > 0:
    X_s, Y_s = train_gen[0]
    print(f'\nX shape  : {X_s.shape}')    # (8, 150, 232)
    print(f'Y[0]     : {Y_s[0]}')
    # Verify feature regions are populated
    print(f'\nFeature region activity:')
    print(f'  Body  [0 :50 ] mean abs: {np.abs(X_s[:,:,0:50]).mean():.4f}')
    print(f'  LHand [50:92 ] mean abs: {np.abs(X_s[:,:,50:92]).mean():.4f}')
    print(f'  RHand [92:134] mean abs: {np.abs(X_s[:,:,92:134]).mean():.4f}')
    print(f'  Face  [134:232] mean abs:{np.abs(X_s[:,:,134:232]).mean():.4f}')


✅ 6,000 rows  ←  how2sign_train_subset.csv
   Feature dim : 232
   Seq length  : 424
✅ 1,739 rows  ←  how2sign_val_subset.csv
   Feature dim : 232
   Seq length  : 424
✅ 2,000 rows  ←  how2sign_test_subset.csv
   Feature dim : 232
   Seq length  : 424

Train batches : 750
Val   batches : 218
Test  batches : 250

X shape  : (8, 424, 232)
Y[0]     : So part of opening the pot is forming your inside of your pot.

Feature region activity:
  Body  [0 :50 ] mean abs: 141.5256
  LHand [50:92 ] mean abs: 249.3575
  RHand [92:134] mean abs: 221.1137
  Face  [134:232] mean abs:197.7548


## 5. Tokenizer — Building the Vocabulary
We use Keras `TextVectorization` to convert raw sentences into token sequences. 
For CTC loss, we reserve index `0` for the `<blank>` token.


In [17]:
# ============================================================
# 5. TOKENIZER
# ============================================================
from tensorflow.keras.layers import TextVectorization
import pickle

# Collect all training sentences
train_df = pd.read_csv(CSV_TRAIN, sep='\t', on_bad_lines='skip')
if 'SENTENCE' not in train_df.columns:
    train_df = pd.read_csv(CSV_TRAIN, on_bad_lines='skip')

train_sentences = train_df['SENTENCE'].fillna('').tolist()

MAX_TOKENS = 3000  # vocab size; training set is 2000 samples
BLANK_INDEX = 0

# Create TextVectorization layer
# We leave index 0 empty for CTC blank
tokenizer = TextVectorization(
    max_tokens=MAX_TOKENS,
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    output_mode='int'
)

# Adapt on training sentences
print("Adapting tokenizer...")
tokenizer.adapt(train_sentences)

vocab = tokenizer.get_vocabulary()
print(f"Vocabulary size: {len(vocab)}")
print(f"Top 10 words: {vocab[:10]}")

# Helper functions for encode/decode
# Shift indices by 1 to reserve 0 for BLANK
def encode_sentence(text):
    indices = tokenizer([text])[0].numpy()
    return indices + 1

def decode_indices(indices):
    words = []
    for idx in indices:
        if idx == 0:
            continue # blank
        idx = idx - 1
        if 0 <= idx < len(vocab):
            w = vocab[idx]
            if w not in ['', '[UNK]']:
                words.append(w)
    return ' '.join(words)

# Smoke test
sample_text = train_sentences[0] if len(train_sentences) > 0 else "hello world"
encoded = encode_sentence(sample_text)
decoded = decode_indices(encoded)
print("Original:", sample_text)
print("Encoded:", encoded)
print("Decoded:", decoded)




Adapting tokenizer...
Vocabulary size: 3000
Top 10 words: ['', '[UNK]', np.str_('the'), np.str_('to'), np.str_('and'), np.str_('you'), np.str_('a'), np.str_('of'), np.str_('that'), np.str_('it')]
Original: So part of opening the pot is forming your inside of your pot.
Encoded: [  15  185    8 1262    3  485   11    2   13  271    8   13  485]
Decoded: so part of opening the pot is your inside of your pot


## 6.5 Data Packer (Fast I/O)
To avoid the 11s/step disk bottleneck, we pack all `.npy` files into a single HDF5 dataset. Run this once locally before uploading.


In [ ]:
# ============================================================
# 6.5 HDF5 DATA PACKER
# ============================================================
import os
import h5py
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
h5_path = Path('/kaggle/working/train_features.h5')
csv_path = CSV_TRAIN   # use the filtered subset, not the full 31k
base_path = BASE_DIR / 'train_features'
if not os.path.exists(h5_path) and os.path.exists(csv_path):
    print("Packing .npy files into HDF5 for fast I/O...")
    df = pd.read_csv(csv_path)
    with h5py.File(h5_path, 'w') as hf:
        for idx, row in tqdm(df.iterrows(), total=len(df)):
            name = str(row.get('SENTENCE_NAME', ''))
            npy_file = os.path.join(base_path, f"{name}.npy")
            if os.path.exists(npy_file):
                data = np.load(npy_file)
                hf.create_dataset(name, data=data, compression="gzip", compression_opts=1)
    print("Packing complete! Generator will now use HDF5.")
else:
    print("HDF5 file already exists or CSV not found. Ready.")


HDF5 file already exists or CSV not found. Ready.


In [21]:
# ============================================================
# 7. CTC DATA GENERATOR
# ============================================================
import h5py
import os
import numpy as np
import tensorflow as tf
class How2SignCTCGenerator(tf.keras.utils.Sequence):
    def __init__(self, csv_path, base_path, batch_size=32, sequence_length=SEQUENCE_LENGTH, max_label_len=50, augment=False):
        import pandas as pd
        if not str(csv_path).endswith('.csv') and not os.path.exists(csv_path):
            self.df = pd.DataFrame()
        else:
            try:
                self.df = pd.read_csv(csv_path, sep='\t', on_bad_lines='skip')
                if 'SENTENCE_NAME' not in self.df.columns:
                    self.df = pd.read_csv(csv_path, on_bad_lines='skip')
            except:
                self.df = pd.DataFrame()
        self.base_path = base_path
        self.batch_size = batch_size
        self.sequence_length = sequence_length
        self.max_label_len = max_label_len
        self.augment = augment
        
        self.h5_path = str(BASE_DIR / 'train_features.h5')
        self.use_h5 = os.path.exists(self.h5_path)
        if self.use_h5:
            self.h5_file = h5py.File(self.h5_path, 'r')
            
    def load_frames(self, video_name):
        if self.use_h5 and video_name in self.h5_file:
            return self.h5_file[video_name][:]
        else:
            p = os.path.join(self.base_path, f"{video_name}.npy")
            return np.load(p) if os.path.exists(p) else np.zeros((1, 232))
    def __len__(self):
        return int(np.ceil(len(self.df) / float(self.batch_size)))
        
    def __getitem__(self, idx):
        batch = self.df.iloc[idx * self.batch_size:(idx + 1) * self.batch_size]
        X = np.zeros((len(batch), self.sequence_length, 232), dtype=np.float32)
        Y = np.zeros((len(batch), self.max_label_len), dtype=np.int32)
        input_lengths = np.zeros((len(batch), 1), dtype=np.int32)
        label_lengths = np.zeros((len(batch), 1), dtype=np.int32)
        
        for i, (_, row) in enumerate(batch.iterrows()):
            name = str(row.get('SENTENCE_NAME', ''))
            frames = self.load_frames(name)
            
            T = min(len(frames), self.sequence_length)
            if T == 0: T = 1
            else:
                data = frames[:T]
                # Augmentation
                if self.augment:
                    data = data + np.random.normal(0, 0.005, data.shape) # Spatial Jitter
                    data = data * np.random.uniform(0.95, 1.05) # Scale shift
                X[i, :T, :] = data
                
            sentence = str(row.get('SENTENCE', ''))
            encoded = encode_sentence(sentence)
            L = min(len(encoded), self.max_label_len)
            
            if L > T: L = T
            # Safety: clamp indices to valid CTC range
            if L > 0:
                safe = np.clip(encoded[:L], 1, MAX_TOKENS)  # 1=min (blank=0 reserved), MAX_TOKENS=max
                Y[i, :L] = safe
            if L == 0: L = 1  # CTC crashes on 0-length labels
            
            input_lengths[i, 0] = max(T, L + 1)  # input must be > label length for CTC
            label_lengths[i, 0] = L
            
            inputs = {
            "input":        X,
            "labels":       Y,
            "input_length": input_lengths,
            "label_length": label_lengths,
        }
        return inputs, np.zeros((len(batch),), dtype=np.float32)
train_gen_ctc = How2SignCTCGenerator(CSV_TRAIN, BASE_DIR / 'train_features', batch_size=32, sequence_length=SEQUENCE_LENGTH, augment=True)
val_gen_ctc = How2SignCTCGenerator(CSV_VAL, BASE_DIR / 'val_features', batch_size=32, sequence_length=SEQUENCE_LENGTH, augment=False)


## 7. CTC-Ready Data Generator
The data generator must yield inputs and label sequences, along with their lengths, to be used by CTC loss.


## 8. CTC BiLSTM Architecture
We use a custom CTC layer to compute the loss during training.


In [ ]:
# ============================================================
# 8. CTC MODEL ARCHITECTURE  (Keras 3 / TF2 compatible)
# ============================================================
import tensorflow as tf
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization,
    LSTM, Bidirectional, Input,
    MultiHeadAttention, LayerNormalization,
    Softmax, Lambda, Activation
)
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K

vocab_size_ctc = MAX_TOKENS + 1
print(f"CTC output classes (incl. blank): {vocab_size_ctc}")

# ── Custom CTC Loss Layer (Keras 3 compatible) ───────────────
class CTCLossLayer(tf.keras.layers.Layer):
    """
    Keras 3 compatible CTC loss layer.
    Wraps tf.nn.ctc_loss as a proper Keras Layer.
    """
    def __init__(self, blank_index=0, **kwargs):
        super().__init__(**kwargs)
        self.blank_index = blank_index

    def call(self, inputs):
        y_pred, labels, input_length, label_length = inputs

        # ✅ Use log-softmax for numerical stability
        # y_pred is already softmax → take log
        log_probs = tf.math.log(y_pred + 1e-8)

        # CTC expects [T, B, C] (time-major)
        log_probs_tm = tf.transpose(log_probs, [1, 0, 2])

        # Flatten length inputs: (B,1) → (B,)
        input_len_1d = tf.cast(tf.reshape(input_length, [-1]), tf.int32)
        label_len_1d = tf.cast(tf.reshape(label_length, [-1]), tf.int32)

        # ✅ Guard: label_length must be <= input_length
        label_len_1d = tf.minimum(label_len_1d, input_len_1d)
        label_len_1d = tf.maximum(label_len_1d, 1)

        loss = tf.nn.ctc_loss(
            labels=labels,
            logits=log_probs_tm,
            label_length=label_len_1d,
            logit_length=input_len_1d,
            logits_time_major=True,
            blank_index=self.blank_index
        )

        # Filter non-finite losses
        loss = tf.where(tf.math.is_finite(loss), loss, tf.zeros_like(loss))

        return tf.reduce_mean(loss)

    def get_config(self):
        config = super().get_config()
        config.update({'blank_index': self.blank_index})
        return config


# ── Inputs ───────────────────────────────────────────────────
input_frames = Input(shape=(SEQUENCE_LENGTH, 232), name='input')
labels_in    = Input(shape=(None,), dtype='int32',  name='labels')
input_len_in = Input(shape=(1,),    dtype='int32',  name='input_length')
label_len_in = Input(shape=(1,),    dtype='int32',  name='label_length')

# ── Feature Encoder ──────────────────────────────────────────
x1 = Bidirectional(LSTM(256, return_sequences=True), name='bilstm_1')(input_frames)
x1 = BatchNormalization(name='bn_1')(x1)

# ✅ MultiHeadAttention used as a Keras layer (correct usage)
attn_out = MultiHeadAttention(num_heads=4, key_dim=64, name='mha')(x1, x1)
x        = LayerNormalization(name='ln_1')(x1 + attn_out)
x        = Dropout(0.3, name='drop_1')(x)

x        = Bidirectional(LSTM(256, return_sequences=True), name='bilstm_2')(x)
x        = BatchNormalization(name='bn_2')(x)
x        = Dropout(0.3, name='drop_2')(x)

# ✅ Dense logits — no softmax here
logits   = Dense(vocab_size_ctc, name='logits')(x)        # (B, T, C)

# ✅ Use Keras Softmax LAYER — not tf.nn.softmax() directly
y_pred   = Softmax(name='prediction')(logits)             # (B, T, C)

# ── CTC Loss (proper Keras layer) ────────────────────────────
ctc_loss_out = CTCLossLayer(blank_index=0, name='ctc_loss')(
    [y_pred, labels_in, input_len_in, label_len_in]
)

# ── Training Model ───────────────────────────────────────────
model_ctc_train = Model(
    inputs  = [input_frames, labels_in, input_len_in, label_len_in],
    outputs = ctc_loss_out,
    name    = 'ctc_train'
)

model_ctc_train.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=lambda y_true, y_pred: y_pred   # the model output IS the CTC loss scalar
)

model_ctc_train.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=5.0)
    # No loss= argument needed — add_loss handles it
)

# ── Inference Model ──────────────────────────────────────────
model_inference = Model(
    inputs  = input_frames,
    outputs = y_pred,
    name    = 'ctc_inference'
)

model_ctc_train.summary()
model_inference.summary()


CTC output classes (incl. blank): 3001


Model: "ctc_train"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 424, 232)  │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_1            │ (None, 424, 512)  │  1,001,472 │ input[0][0]       │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_1                │ (None, 424, 512)  │      2,048 │ bilstm_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mha                 │ (None, 424, 512)  │    525,568 │ bn_1[0][0],       │
│ (MultiHeadAttentio… │                   │            │ bn_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 424, 512)  │          0 │ bn_1[0][0],       │
│                     │                   │            │ mha[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ln_1                │ (None, 424, 512)  │      1,024 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_1 (Dropout)    │ (None, 424, 512)  │          0 │ ln_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_2            │ (None, 424, 512)  │  1,574,912 │ drop_1[0][0]      │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_2                │ (None, 424, 512)  │      2,048 │ bilstm_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_2 (Dropout)    │ (None, 424, 512)  │          0 │ bn_2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ logits (Dense)      │ (None, 424, 3001) │  1,539,513 │ drop_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ prediction          │ (None, 424, 3001) │          0 │ logits[0][0]      │
│ (Softmax)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ labels (InputLayer) │ (None, None)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_length        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ label_length        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ctc_loss            │ ()                │          0 │ prediction[0][0], │
│ (CTCLossLayer)      │                   │            │ labels[0][0],     │
│                     │                   │            │ input_length[0][… │
│                     │                   │            │ label_length[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,646,585 (17.73 MB)

 Trainable params: 4,644,537 (17.72 MB)

...

Model: "ctc_inference"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 424, 232)  │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_1            │ (None, 424, 512)  │  1,001,472 │ input[0][0]       │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_1                │ (None, 424, 512)  │      2,048 │ bilstm_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mha                 │ (None, 424, 512)  │    525,568 │ bn_1[0][0],       │
│ (MultiHeadAttentio… │                   │            │ bn_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 424, 512)  │          0 │ bn_1[0][0],       │
│                     │                   │            │ mha[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ln_1                │ (None, 424, 512)  │      1,024 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_1 (Dropout)    │ (None, 424, 512)  │          0 │ ln_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_2            │ (None, 424, 512)  │  1,574,912 │ drop_1[0][0]      │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_2                │ (None, 424, 512)  │      2,048 │ bilstm_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_2 (Dropout)    │ (None, 424, 512)  │          0 │ bn_2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ logits (Dense)      │ (None, 424, 3001) │  1,539,513 │ drop_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ prediction          │ (None, 424, 3001) │          0 │ logits[0][0]      │
│ (Softmax)           │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,646,585 (17.73 MB)

 Trainable params: 4,644,537 (17.72 MB)

 Non-trainable params: 2,048 (8.00 KB)

## 9. Training
Training with EarlyStopping, ReduceLROnPlateau, and ModelCheckpoint for 30 epochs.


In [ ]:
# ============================================================
# 9. TRAINING LOOP
# ============================================================
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=7,   # was 5 — give it more room
                  restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', 
                      factor=0.5, 
                      patience=3,                    # was 2 — less trigger-happy
                      min_lr=1e-6),
    ModelCheckpoint('best_cslr_model.weights.h5', 
                    save_best_only=True, 
                    save_weights_only=False)
]

history = model_ctc_train.fit(
     train_gen_ctc,
     validation_data=val_gen_ctc,
     epochs=10,
     callbacks=callbacks
 )

print("Model training ready (epochs=30).")



Epoch 1/10
 18/188 ━━━━━━━━━━━━━━━━━━━━ 3:19 1s/step - loss: 954.0554

## 10. Evaluation & Metrics
Calculating Word Error Rate (WER), Precision, Recall, F1, and PR-AUC.


In [ ]:
pip install jiwer


In [1]:
# ============================================================
# 10. EVALUATION & METRICS
# ============================================================
# !pip install jiwer
import jiwer
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
import tensorflow.keras.backend as K
def ctc_greedy_decode(logits, input_lengths):
    decoded, _ = tf.keras.backend.ctc_decode(logits, input_length=input_lengths, greedy=True)
    return decoded[0].numpy()
def evaluate_model(model_inf, test_gen):
    print("Evaluating model...")
    all_hypotheses = []
    all_references = []
    
    max_batches = len(test_gen)  # full test set
    for i in range(max_batches):
        batch_x, _ = test_gen[i]
        logits = model_inf.predict(batch_x['input'], verbose=0)
        decoded = ctc_greedy_decode(logits, batch_x['input_length'].flatten())
        
        for j in range(len(decoded)):
            hyp_indices = [idx for idx in decoded[j] if idx != -1]
            hyp_str = decode_indices(hyp_indices)
            all_hypotheses.append(hyp_str)
            
            ref_indices = [idx for idx in batch_x['labels'][j] if idx != 0]
            ref_str = decode_indices(ref_indices)
            all_references.append(ref_str)
            
    all_hypotheses = [h if len(h) > 0 else "empty" for h in all_hypotheses]
    all_references = [r if len(r) > 0 else "empty" for r in all_references]
    
    wer = jiwer.wer(all_references, all_hypotheses)
    print(f"Word Error Rate (WER): {wer:.4f}")
    
    return all_references, all_hypotheses
test_gen_ctc = How2SignCTCGenerator(CSV_TEST, BASE_DIR / 'test_features', batch_size=32, sequence_length=SEQUENCE_LENGTH, augment=False)
# refs, hyps = evaluate_model(model_inference, test_gen_ctc)


ModuleNotFoundError: No module named 'jiwer'

## 11. Stabilization Tracker
A sliding window approach with majority voting to smooth out raw CTC predictions in real-time.


In [ ]:
# ============================================================
# 11. STABILIZATION TRACKER
# ============================================================
from collections import deque
import time

class StabilizationTracker:
    def __init__(self, window_size=15, majority_ratio=0.6, cooldown_s=1.0):
        self.window_size = window_size
        self.majority_ratio = majority_ratio
        self.cooldown_s = cooldown_s
        self.buffer = deque(maxlen=window_size)
        self.last_commit_time = 0
        self.last_committed_word = ""
        self.sentence = []

    def update(self, predicted_word):
        if not predicted_word:
            self.buffer.append(None)
            return None
            
        self.buffer.append(predicted_word)
        
        if len(self.buffer) < self.window_size:
            return None
            
        counts = {}
        for w in self.buffer:
            if w: counts[w] = counts.get(w, 0) + 1
            
        if not counts: return None
        
        top_word = max(counts, key=counts.get)
        top_ratio = counts[top_word] / self.window_size
        
        now = time.time()
        if top_ratio >= self.majority_ratio:
            if top_word != self.last_committed_word and (now - self.last_commit_time) > self.cooldown_s:
                self.sentence.append(top_word)
                self.last_committed_word = top_word
                self.last_commit_time = now
                self.buffer.clear()
                return top_word
        return None

tracker = StabilizationTracker()



## 12. Real-Time MediaPipe Webcam Inference
Webcam demo utilizing MediaPipe Holistic. We extract landmarks, match OpenPose indices, buffer frames, predict with the trained sequence model, and stabilize the output.
Both modes (live webcam and offline video file) are supported. Minimal latency is ensured by sliding the buffer window.


In [ ]:
pip install mediapipe==0.10.14


In [3]:
# ============================================================
# 12. WEBCAM INFERENCE LOOP  (232-dim, dynamic SEQUENCE_LENGTH)
# ============================================================
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
from collections import deque

mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

# ── Face keypoints (same 49 indices used in Section 2.5) ────
_FACE_KP_INDICES = (
    [17,18,19,20,21] + [22,23,24,25,26] +     # eyebrows (10)
    [36,37,38,39,40,41] + [42,43,44,45,46,47] + # eyes (12)
    [68,69] +                                    # pupils (2)
    [27,28,29,30] + [33] +                       # nose bridge+tip (5)
    [48,49,50,51,52,53,54,55,56,57,58,59] +      # outer lips (12)
    [60,61,62,63,64,65,66,67]                     # inner lips (8)
)  # 49 total

# MediaPipe Face Mesh → OpenPose-70 approximate mapping
# (MediaPipe has 468 mesh points; we pick the closest ones)
_MP_FACE_TO_OP70 = {
    17:70, 18:63, 19:105, 20:66, 21:107,           # R eyebrow
    22:336, 23:296, 24:334, 25:293, 26:300,         # L eyebrow
    36:33, 37:160, 38:158, 39:133, 40:153, 41:144,  # R eye
    42:362, 43:385, 44:387, 45:263, 46:373, 47:380, # L eye
    68:468, 69:473,                                  # pupils (iris)
    27:6, 28:197, 29:195, 30:5,                      # nose bridge
    33:1,                                            # nose tip
    48:61, 49:185, 50:40, 51:39, 52:37, 53:0,       # outer lips R
    54:267, 55:269, 56:270, 57:409, 58:291, 59:375, # outer lips L
    60:78, 61:191, 62:80, 63:81,                     # inner lips top
    64:311, 65:310, 66:415, 67:308                   # inner lips bottom
}


def mediapipe_to_232dim(results):
    """
    Convert MediaPipe Holistic output → (232,) float32 vector.
    Layout matches training: body(50) + lhand(42) + rhand(42) + face(98) = 232
    """
    # ── Body: 25 OpenPose joints × (x,y) = 50 ──────────────
    pose = np.zeros((25, 2), dtype=np.float32)
    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        def set_p(op_idx, mp_idx):
            if mp_idx < len(lm):
                pose[op_idx] = [lm[mp_idx].x, lm[mp_idx].y]

        set_p(0, 0)    # Nose
        set_p(2, 12)   # RShoulder
        set_p(3, 14)   # RElbow
        set_p(4, 16)   # RWrist
        set_p(5, 11)   # LShoulder
        set_p(6, 13)   # LElbow
        set_p(7, 15)   # LWrist
        set_p(9, 24)   # RHip
        set_p(10, 26)  # RKnee
        set_p(11, 28)  # RAnkle
        set_p(12, 23)  # LHip
        set_p(13, 25)  # LKnee
        set_p(14, 27)  # LAnkle
        set_p(15, 5)   # REye
        set_p(16, 2)   # LEye
        set_p(17, 8)   # REar
        set_p(18, 7)   # LEar
        set_p(19, 31)  # LBigToe
        set_p(21, 29)  # LHeel
        set_p(22, 32)  # RBigToe
        set_p(24, 30)  # RHeel

        # Interpolate Neck (1) and MidHip (8)
        if pose[2].any() and pose[5].any():
            pose[1] = (pose[2] + pose[5]) / 2.0
        if pose[9].any() and pose[12].any():
            pose[8] = (pose[9] + pose[12]) / 2.0

    # ── Hands: 21 joints × (x,y) each = 42 + 42 ───────────
    left_hand = np.zeros((21, 2), dtype=np.float32)
    if results.left_hand_landmarks:
        for k, pt in enumerate(results.left_hand_landmarks.landmark):
            left_hand[k] = [pt.x, pt.y]

    right_hand = np.zeros((21, 2), dtype=np.float32)
    if results.right_hand_landmarks:
        for k, pt in enumerate(results.right_hand_landmarks.landmark):
            right_hand[k] = [pt.x, pt.y]

    # ── Face: 49 selected keypoints × (x,y) = 98 ──────────
    face = np.zeros((49, 2), dtype=np.float32)
    if results.face_landmarks:
        mesh = results.face_landmarks.landmark
        for k, op_idx in enumerate(_FACE_KP_INDICES):
            mp_idx = _MP_FACE_TO_OP70.get(op_idx)
            if mp_idx is not None and mp_idx < len(mesh):
                face[k] = [mesh[mp_idx].x, mesh[mp_idx].y]

    return np.concatenate([
        pose.flatten(),        #  50
        left_hand.flatten(),   #  42
        right_hand.flatten(),  #  42
        face.flatten()         #  98
    ])                         # 232 total


def run_inference(source=0):
    cap = cv2.VideoCapture(source)
    tracker = StabilizationTracker(window_size=15, majority_ratio=0.6)
    frame_buffer = deque(maxlen=SEQUENCE_LENGTH)

    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:
        frame_count = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(image)
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            features = mediapipe_to_232dim(results)
            frame_buffer.append(features)

            # Sliding window: infer every 5 frames for low latency
            if len(frame_buffer) > 10 and frame_count % 5 == 0:
                X = np.expand_dims(np.stack(frame_buffer), axis=0)
                if X.shape[1] < SEQUENCE_LENGTH:
                    pad = np.zeros((1, SEQUENCE_LENGTH - X.shape[1], 232))
                    X = np.concatenate([X, pad], axis=1)

                preds = model_inference.predict(X, verbose=0)
                input_lengths = np.array([min(len(frame_buffer), SEQUENCE_LENGTH)])
                decoded, _ = tf.keras.backend.ctc_decode(
                    preds, input_length=input_lengths, greedy=True)
                indices = decoded[0][0].numpy()

                sentence = decode_indices(indices)
                tracker.update(sentence)

            final_text = ' '.join(tracker.sentence) if tracker.sentence else ''
            cv2.putText(image, final_text, (10, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.imshow('Sign Language Translation', image)

            frame_count += 1
            if cv2.waitKey(10) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()

# run_inference(0)



## 13. Export to ONNX
We convert the inference model to ONNX format for deployment.

In [ ]:
pip install tf2onnx


In [ ]:
# ============================================================
# 13. EXPORT TO ONNX  (SavedModel-first for BiLSTM stability)
# ============================================================
import tensorflow as tf
import tf2onnx
import onnx
import os

# Load best weights
try:
    model_inference.load_weights('best_cslr_model.weights.h5')
    print("Loaded best weights.")
except Exception as e:
    print("Could not load weights:", e)

onnx_path = "cslr_inference_model.onnx"

# Strategy 1: SavedModel → tf2onnx  (more stable for BiLSTM)
saved_model_dir = 'cslr_saved_model'
try:
    model_inference.save(saved_model_dir)
    print("Saved to SavedModel format. Converting to ONNX...")
    os.system(
        f'python -m tf2onnx.convert '
        f'--saved-model {saved_model_dir} '
        f'--output {onnx_path} '
        f'--opset 13'
    )
    print(f"ONNX model saved to {onnx_path}")
except Exception as e1:
    print(f"SavedModel strategy failed ({e1}), trying Keras direct...")
    try:
        input_signature = [tf.TensorSpec(
            [1, SEQUENCE_LENGTH, 232], tf.float32, name='input')]
        onnx_model, _ = tf2onnx.convert.from_keras(
            model_inference, input_signature, opset=13)
        with open(onnx_path, 'wb') as f:
            f.write(onnx_model.SerializeToString())
        print(f"ONNX model saved to {onnx_path} (Keras direct fallback)")
    except Exception as e2:
        print(f"Both strategies failed. Error: {e2}")

# Optional: INT8 quantization for ~2x faster inference
try:
    import onnxruntime
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quantize_dynamic(onnx_path, 'cslr_int8.onnx', weight_type=QuantType.QInt8)
    print("INT8 quantized model saved to cslr_int8.onnx")
except Exception as e:
    print(f"INT8 quantization skipped: {e}")



## 14. Package & Download
Zips the extracted `.npy` feature folders and the filtered CSV files into a single archive so you can download everything from Kaggle in one click.


In [ ]:
# ============================================================
# 14. PACKAGE FEATURES + CSVs FOR DOWNLOAD
# ============================================================
import os, zipfile, glob
from pathlib import Path

def zip_folder(folder, zip_handle, arcname_prefix):
    """Add every file inside `folder` to the zip under `arcname_prefix/`."""
    for fpath in sorted(glob.glob(os.path.join(folder, '**', '*'), recursive=True)):
        if os.path.isfile(fpath):
            arcname = arcname_prefix + '/' + os.path.relpath(fpath, folder)
            zip_handle.write(fpath, arcname)

OUTPUT_ZIP = '/kaggle/working/how2sign_features_export.zip'

# Folders to pack
FEATURE_DIRS = {
    'train_features': '/kaggle/working/train_features',
    'val_features'  : '/kaggle/working/val_features',
    'test_features' : '/kaggle/working/test_features',
}

# Subset CSVs to include
CSV_FILES = {
    'how2sign_train_subset.csv': '/kaggle/working/how2sign_train_subset.csv',
    'how2sign_val_subset.csv'  : '/kaggle/working/how2sign_val_subset.csv',
    'how2sign_test_subset.csv' : '/kaggle/working/how2sign_test_subset.csv',
}

print(f'Building archive  →  {OUTPUT_ZIP}')
print('This may take a few minutes depending on how many .npy files were extracted...\n')

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:

    # ── Feature .npy folders ─────────────────────────────────
    for arc_prefix, folder_path in FEATURE_DIRS.items():
        if os.path.isdir(folder_path):
            files = glob.glob(os.path.join(folder_path, '*.npy'))
            print(f'  📦  {arc_prefix}/  →  {len(files):,} .npy files')
            zip_folder(folder_path, zf, arc_prefix)
        else:
            print(f'  ⚠️   {arc_prefix}/ not found — skipping')

    # ── Subset CSVs ──────────────────────────────────────────
    for arc_name, csv_path in CSV_FILES.items():
        if os.path.isfile(csv_path):
            zf.write(csv_path, arc_name)
            size_kb = os.path.getsize(csv_path) / 1024
            print(f'  📄  {arc_name}  ({size_kb:.1f} KB)')
        else:
            print(f'  ⚠️   {arc_name} not found — skipping')

zip_size_mb = os.path.getsize(OUTPUT_ZIP) / (1024 * 1024)
print(f'\n✅  Done!  Archive size: {zip_size_mb:.1f} MB')
print(f'   →  {OUTPUT_ZIP}')
print('\nIn the Kaggle sidebar, go to Output → how2sign_features_export.zip → Download')



In [ ]:
# ============================================================
# 12. FULL WEBCAM INFERENCE SCRIPT
# ============================================================
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
import time
from collections import deque

# ── 1. Stabilization Tracker (Smoothes out the text) ────────
class StabilizationTracker:
    def __init__(self, window_size=15, majority_ratio=0.6, cooldown_s=1.0):
        self.window_size = window_size
        self.majority_ratio = majority_ratio
        self.cooldown_s = cooldown_s
        self.buffer = deque(maxlen=window_size)
        self.last_commit_time = 0
        self.last_committed_word = ""
        self.sentence = []

    def update(self, predicted_word):
        if not predicted_word:
            self.buffer.append(None)
            return None
            
        self.buffer.append(predicted_word)
        
        if len(self.buffer) < self.window_size:
            return None
            
        counts = {}
        for w in self.buffer:
            if w: counts[w] = counts.get(w, 0) + 1
            
        if not counts: return None
        
        top_word = max(counts, key=counts.get)
        top_ratio = counts[top_word] / self.window_size
        
        now = time.time()
        if top_ratio >= self.majority_ratio:
            if top_word != self.last_committed_word and (now - self.last_commit_time) > self.cooldown_s:
                self.sentence.append(top_word)
                self.last_committed_word = top_word
                self.last_commit_time = now
                self.buffer.clear()
                return top_word
        return None

# ── 2. MediaPipe to OpenPose Mappings ───────────────────────
mp_holistic = mp.solutions.holistic

_FACE_KP_INDICES = (
    [17,18,19,20,21] + [22,23,24,25,26] +     # eyebrows (10)
    [36,37,38,39,40,41] + [42,43,44,45,46,47] + # eyes (12)
    [68,69] +                                    # pupils (2)
    [27,28,29,30] + [33] +                       # nose bridge+tip (5)
    [48,49,50,51,52,53,54,55,56,57,58,59] +      # outer lips (12)
    [60,61,62,63,64,65,66,67]                     # inner lips (8)
)  

_MP_FACE_TO_OP70 = {
    17:70, 18:63, 19:105, 20:66, 21:107, 
    22:336, 23:296, 24:334, 25:293, 26:300, 
    36:33, 37:160, 38:158, 39:133, 40:153, 41:144, 
    42:362, 43:385, 44:387, 45:263, 46:373, 47:380, 
    68:468, 69:473, 27:6, 28:197, 29:195, 30:5, 33:1, 
    48:61, 49:185, 50:40, 51:39, 52:37, 53:0, 
    54:267, 55:269, 56:270, 57:409, 58:291, 59:375, 
    60:78, 61:191, 62:80, 63:81, 64:311, 65:310, 66:415, 67:308                   
}

# ── 3. Feature Extraction (232-dim) ─────────────────────────
def mediapipe_to_232dim(results):
    pose = np.zeros((25, 2), dtype=np.float32)
    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        def set_p(op_idx, mp_idx):
            if mp_idx < len(lm):
                pose[op_idx] = [lm[mp_idx].x, lm[mp_idx].y]

        set_p(0, 0); set_p(2, 12); set_p(3, 14); set_p(4, 16)
        set_p(5, 11); set_p(6, 13); set_p(7, 15); set_p(9, 24)
        set_p(10, 26); set_p(11, 28); set_p(12, 23); set_p(13, 25)
        set_p(14, 27); set_p(15, 5); set_p(16, 2); set_p(17, 8)
        set_p(18, 7); set_p(19, 31); set_p(21, 29); set_p(22, 32)
        set_p(24, 30)

        if pose[2].any() and pose[5].any():
            pose[1] = (pose[2] + pose[5]) / 2.0
        if pose[9].any() and pose[12].any():
            pose[8] = (pose[9] + pose[12]) / 2.0

    left_hand = np.zeros((21, 2), dtype=np.float32)
    if results.left_hand_landmarks:
        for k, pt in enumerate(results.left_hand_landmarks.landmark):
            left_hand[k] = [pt.x, pt.y]

    right_hand = np.zeros((21, 2), dtype=np.float32)
    if results.right_hand_landmarks:
        for k, pt in enumerate(results.right_hand_landmarks.landmark):
            right_hand[k] = [pt.x, pt.y]

    face = np.zeros((49, 2), dtype=np.float32)
    if results.face_landmarks:
        mesh = results.face_landmarks.landmark
        for k, op_idx in enumerate(_FACE_KP_INDICES):
            mp_idx = _MP_FACE_TO_OP70.get(op_idx)
            if mp_idx is not None and mp_idx < len(mesh):
                face[k] = [mesh[mp_idx].x, mesh[mp_idx].y]

    return np.concatenate([pose.flatten(), left_hand.flatten(), right_hand.flatten(), face.flatten()])

# ── 4. Main Inference Loop ──────────────────────────────────
def run_inference(source=0):
    print("🎥 Starting Webcam... Press 'q' in the video window to stop.")
    cap = cv2.VideoCapture(source)
    tracker = StabilizationTracker(window_size=15, majority_ratio=0.6)
    
    # We use SEQUENCE_LENGTH from your earlier cells (likely 424)
    frame_buffer = deque(maxlen=SEQUENCE_LENGTH)

    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:
        frame_count = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                print("❌ Failed to grab frame. Check your webcam.")
                break

            # Process with MediaPipe
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(image)
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            # Extract features and add to buffer
            features = mediapipe_to_232dim(results)
            frame_buffer.append(features)

            # Run prediction every 5 frames to keep it real-time
            if len(frame_buffer) > 10 and frame_count % 5 == 0:
                X = np.expand_dims(np.stack(frame_buffer), axis=0)
                
                # Pad the sequence if it hasn't reached SEQUENCE_LENGTH yet
                if X.shape[1] < SEQUENCE_LENGTH:
                    pad = np.zeros((1, SEQUENCE_LENGTH - X.shape[1], 232))
                    X = np.concatenate([X, pad], axis=1)

                # Predict!
                preds = model_inference.predict(X, verbose=0)
                input_lengths = np.array([min(len(frame_buffer), SEQUENCE_LENGTH)])
                
                decoded, _ = tf.keras.backend.ctc_decode(
                    preds, input_length=input_lengths, greedy=True)
                indices = decoded[0][0].numpy()

                # Translate indices back to English words
                sentence = decode_indices(indices)
                tracker.update(sentence)

            # Draw the translated text on the screen
            final_text = ' '.join(tracker.sentence) if tracker.sentence else '...'
            
            # Draw a black background rectangle for the text to make it readable
            cv2.rectangle(image, (0, 0), (640, 80), (0, 0, 0), -1)
            cv2.putText(image, final_text, (10, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            
            cv2.imshow('Sign Language Translation (Press Q to quit)', image)

            frame_count += 1
            # Press 'q' to quit
            if cv2.waitKey(10) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()
    print("🛑 Webcam closed.")

# ── 5. START! ───────────────────────────────────────────────
run_inference(0)
